# 06 · LLM benchmark for report labelling (Kaggle, 2× T4)

**Question:** is the report-labelling result specific to Qwen, or do other open LLMs do as well?

Every model gets the **same prompt** (`kneemri.labels.llm`: system prompt + 0–3 rubric, JSON answer), the **same
decoding** (greedy, expected grade from the grade-digit probabilities) and **4-bit NF4** weights. Only the model ID
changes. Scores are ROC-AUC against the **58 radiologist-labelled gold studies**, with study-level bootstrap 95% CIs.

| Model | Hugging Face ID |
|---|---|
| Llama 3.1 8B Instruct | `meta-llama/Llama-3.1-8B-Instruct` |
| Gemma 2 9B Instruct | `google/gemma-2-9b-it` |
| Mistral 7B Instruct v0.3 | `mistralai/Mistral-7B-Instruct-v0.3` |

Qwen2.5-14B (the model used in the pipeline) and the Claude rubric labels are added to the comparison table in
notebook 05 from their saved grades.

*A model that fails to load (for example a gated one without licence access) is skipped and logged; the run continues.*

**Before running:** accept the three model licences on Hugging Face, add your token as a Kaggle secret named
`HF` or `HF_TOKEN` (Add-ons → Secrets), attach the competition data, enable **GPU T4 ×2** and **Internet**.

In [ ]:
!git clone -q -b llm-benchmark https://github.com/lucky07-07/knee-mri-abnormality-detection.git /kaggle/working/repo
!pip install -q -e /kaggle/working/repo
!pip install -q -U "transformers>=4.44" accelerate "bitsandbytes>=0.43"

In [ ]:
import os, glob, json, subprocess, threading, time
import pandas as pd, torch
from kaggle_secrets import UserSecretsClient

MODE = "gold"   # "gold": the 58 gold reports (~15-20 min in total) | "full": all 4,407 reports (~1.5-2.5 h per model)

MODELS = {
    "llama31_8b": dict(model_id="meta-llama/Llama-3.1-8B-Instruct"),
    # Gemma 2: eager attention (soft-capping) and fp32 compute, fp16 activations can overflow on T4
    "gemma2_9b":  dict(model_id="google/gemma-2-9b-it", attn_implementation="eager", compute_dtype="float32"),
    "mistral_7b": dict(model_id="mistralai/Mistral-7B-Instruct-v0.3"),
}

# depth-limited search: a recursive glob would walk the 820K-file DICOM tree
COMP = os.path.dirname(sorted(f for d in (1, 2, 3) for f in glob.glob(f"/kaggle/input/{'/'.join(['*'] * d)}/train.csv"))[0])
OUT = "/kaggle/working/benchmark"
os.makedirs(OUT, exist_ok=True)
def hf_token():   # the Kaggle secret may be labelled HF_TOKEN or HF; never printed
    for label in ("HF_TOKEN", "HF"):
        try:
            return UserSecretsClient().get_secret(label)
        except Exception:
            pass
    print("WARNING: no HF / HF_TOKEN secret readable: the gated models will fail to download and be skipped")
    return ""
os.environ["HF_TOKEN"] = hf_token()
print("competition data:", COMP, "| GPUs:", torch.cuda.device_count(), "| mode:", MODE)

## 1. Grade the reports: one subprocess per model, models queued round-robin over the GPUs

In [ ]:
%%writefile /kaggle/working/grade_one.py
import json, os, sys, time
import pandas as pd, torch
from kneemri.constants import LABELS
from kneemri.labels.llm import ReportGrader

name, cfg, mode, comp, out = sys.argv[1], json.loads(sys.argv[2]), sys.argv[3], sys.argv[4], sys.argv[5]
train = pd.read_csv(f"{comp}/train.csv")
if mode == "gold":
    train = train[train[LABELS].notna().all(axis=1)]
t0 = time.time()
grader = ReportGrader(token=os.environ.get("HF_TOKEN") or None, **cfg)
load_min = (time.time() - t0) / 60
grades = grader.grade(train.Report.fillna("").tolist(), batch_size=16 if mode == "full" else 8)
grades.insert(0, "StudyInstanceUID", train.StudyInstanceUID.values)
grades.to_csv(f"{out}/grades_{name}_{mode}.csv", index=False)
parse_fail = float(grades[[f"{c}__hard" for c in LABELS]].isna().any(axis=1).mean())
meta = dict(name=name, model_id=cfg["model_id"], mode=mode, n_reports=len(grades), load_minutes=round(load_min, 1),
            grade_minutes=round((time.time() - t0) / 60 - load_min, 1), parse_fail_rate=round(parse_fail, 3),
            peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 1))
json.dump(meta, open(f"{out}/meta_{name}_{mode}.json", "w"), indent=1)
print("DONE", meta, flush=True)

In [ ]:
def run_queue(gpu, names):
    for n in names:
        with open(f"{OUT}/log_{n}_{MODE}.txt", "w") as log:
            rc = subprocess.run(["python", "-u", "/kaggle/working/grade_one.py", n, json.dumps(MODELS[n]), MODE, COMP, OUT],
                                env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT).returncode
        if rc == 0:
            print(f"[GPU {gpu}] {n}: done", flush=True)
        else:   # gated or failed model: log it and carry on with the rest
            err = [l.strip() for l in open(f"{OUT}/log_{n}_{MODE}.txt") if l.strip()][-1:]
            print(f"[GPU {gpu}] SKIPPED {MODELS[n]['model_id']}: {err[0] if err else 'exit code ' + str(rc)}", flush=True)

names, ng = list(MODELS), max(1, torch.cuda.device_count())
t0 = time.time()
threads = [threading.Thread(target=run_queue, args=(g, names[g::ng])) for g in range(ng)]
[t.start() for t in threads]; [t.join() for t in threads]
print(f"all models finished in {(time.time() - t0) / 60:.1f} min")
for n in names:   # last lines of each log (errors show up here)
    print(f"\n===== {n} =====\n" + "".join(open(f"{OUT}/log_{n}_{MODE}.txt").readlines()[-6:]))

## 2. Score against the 58 gold studies (macro AUC with bootstrap 95% CI, per-finding AUC)

In [ ]:
from kneemri.constants import LABELS
from kneemri.labels.benchmark import benchmark_table, grade_scores

train = pd.read_csv(f"{COMP}/train.csv")
gold = train[train[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
sources = {n: grade_scores(pd.read_csv(f"{OUT}/grades_{n}_{MODE}.csv"))
           for n in MODELS if os.path.exists(f"{OUT}/grades_{n}_{MODE}.csv")}
table = benchmark_table(sources, gold, n_boot=2000)
table.to_csv(f"{OUT}/benchmark_gold.csv", index=False)
meta = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/meta_*_{MODE}.json"))])
display(meta)
display(table)

**Reading the table:** `macro_auc` is the mean over findings of the AUC of each model's expected grade against the
gold labels; `ci_low`/`ci_high` is the bootstrap 95% interval. With 58 studies, overlapping intervals mean the models
are not distinguishable. `parse_fail_rate` is the share of reports where at least one grade could not be parsed
(it should be near 0; a high value means the model ignored the JSON format).

Download `benchmark/` from the notebook output; notebook 05 merges it with the Qwen2.5-14B and Claude results.